# DPO Training — Safety-Aligned Distillation

Two steps:
1. **prepare_dpo_data** — run the baseline SFT model on harmful prompts to generate rejected responses
2. **dpo** — train the student model with DPO using the preference pairs

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add your HuggingFace token: Add-ons → Secrets → Add **HF_TOKEN** (toggle On)

## Step 0: Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft trl huggingface_hub bitsandbytes tqdm

## Step 1: Setup — HF token and clone repo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("HuggingFace login successful.")

In [ ]:
# Clone (or pull) the project repo
import os

REPO_URL = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
REPO_DIR = "/kaggle/working/Distill-Safety-Aligned-Models"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## Step 2: Generate DPO Preference Pairs

Runs the **baseline SFT model** on the harmful prompts to produce **rejected** responses.
The teacher's refusals (already in `data/raw_teacher_outputs.jsonl`) are the **chosen** responses.

Output: `data/dpo_pairs.jsonl`
Expected time: ~30–45 min on T4

In [ ]:
# Configuration
STUDENT_MODEL_ID  = "meta-llama/Llama-3.2-1B-Instruct"
SFT_CHECKPOINT    = "outputs/baseline"   # Member 2's checkpoint (already in repo)
RAW_FILE          = "data/raw_teacher_outputs.jsonl"
DPO_PAIRS_OUTPUT  = "data/dpo_pairs.jsonl"
BATCH_SIZE        = 8
MAX_NEW_TOKENS    = 256

print(f"Student model : {STUDENT_MODEL_ID}")
print(f"SFT checkpoint: {SFT_CHECKPOINT}")
print(f"Raw file      : {RAW_FILE}")
print(f"Output        : {DPO_PAIRS_OUTPUT}")

In [ ]:
import json
import re
import torch
from pathlib import Path
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."

REFUSAL_PREFIXES = [
    "i'm sorry", "i can't", "i cannot", "i'm not able", "i won't",
    "i will not", "as an ai", "i'm unable", "i apologize",
    "that's not something i", "i don't think i should",
    "this request involves", "i must decline",
]
REFUSAL_PATTERNS = [
    r"i (am|'m) (not able|unable) to",
    r"i (can't|cannot|won't|will not) (help|assist|provide|generate|create)",
    r"(this|that) (request|prompt|query) (is|seems|appears) (harmful|dangerous|unethical|illegal)",
]

def is_refusal(text: str) -> bool:
    t = text.lower().strip()
    for p in REFUSAL_PREFIXES:
        if t.startswith(p):
            return True
    for pat in REFUSAL_PATTERNS:
        if re.search(pat, t):
            return True
    return False

# Load harmful records with teacher refusals
records = []
with open(RAW_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

harmful_refusals = [
    r for r in records
    if r.get("category") == "harmful" and r.get("is_refusal", False)
]
print(f"Harmful refusal records: {len(harmful_refusals)}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model + SFT LoRA adapter
torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch_dtype,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, SFT_CHECKPOINT)
model.eval()
print("Baseline SFT model loaded.")

In [ ]:
# Generate rejected responses from baseline model
prompts = [r["prompt"]   for r in harmful_refusals]
chosen  = [r["response"] for r in harmful_refusals]  # teacher refusals

rejected = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="generating rejected"):
    batch = prompts[i : i + BATCH_SIZE]
    chats = [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": p}]
        for p in batch
    ]
    inputs_text = [
        tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        for chat in chats
    ]
    inputs = tokenizer(
        inputs_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    for output_ids in outputs:
        resp = tokenizer.decode(output_ids[input_len:], skip_special_tokens=True).strip()
        rejected.append(resp)

print(f"Generated {len(rejected)} rejected responses.")

In [ ]:
# Build and save DPO pairs (skip pairs where baseline also refused)
pairs = []
skipped = 0
for prompt, ch, rej in zip(prompts, chosen, rejected):
    if is_refusal(rej):
        skipped += 1
        continue
    pairs.append({"prompt": prompt, "chosen": ch, "rejected": rej})

print(f"Valid DPO pairs      : {len(pairs)}")
print(f"Skipped (both refuse): {skipped}")

with open(DPO_PAIRS_OUTPUT, "w") as f:
    for pair in pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"Saved → {DPO_PAIRS_OUTPUT}")

# Preview first pair
print("\n--- Example pair ---")
print("PROMPT  :", pairs[0]["prompt"][:80])
print("CHOSEN  :", pairs[0]["chosen"][:80])
print("REJECTED:", pairs[0]["rejected"][:80])

## Step 3: DPO Training

Starts from the merged SFT baseline and trains a fresh LoRA adapter using DPO loss.

Expected time: ~2–3 hours on T4

In [ ]:
# DPO configuration
DPO_OUTPUT_DIR   = "outputs/dpo"
DPO_LOGGING_DIR  = "logs/dpo"
BETA             = 0.1      # KL penalty — how much to stay close to SFT baseline
LEARNING_RATE    = 5e-5     # 4x lower than SFT; DPO destabilises at high LR
NUM_EPOCHS       = 2
BATCH_SIZE_TRAIN = 2        # DPO processes 4 sequences per pair; keep small
GRAD_ACCUM       = 16       # effective batch = 2 * 16 = 32
MAX_PROMPT_LEN   = 512
MAX_LEN          = 1024
LORA_R           = 16
LORA_ALPHA       = 32

In [ ]:
import os
import json
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

set_seed(42)
os.makedirs(DPO_OUTPUT_DIR,  exist_ok=True)
os.makedirs(DPO_LOGGING_DIR, exist_ok=True)

# ── Reload tokenizer (left-padding for DPO)
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Load DPO pairs
dpo_pairs = []
with open(DPO_PAIRS_OUTPUT) as f:
    for line in f:
        line = line.strip()
        if line:
            dpo_pairs.append(json.loads(line))
print(f"DPO pairs loaded: {len(dpo_pairs)}")

# ── Format as TRL messages schema
SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."
rows = []
for pair in dpo_pairs:
    rows.append({
        "prompt":   [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user",   "content": pair["prompt"]}],
        "chosen":   [{"role": "assistant", "content": pair["chosen"]}],
        "rejected": [{"role": "assistant", "content": pair["rejected"]}],
    })
dpo_dataset = Dataset.from_list(rows)
print(f"Dataset size: {len(dpo_dataset)}")

In [ ]:
# ── Load base model, merge SFT LoRA, attach fresh DPO LoRA
torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch_dtype,
    device_map="auto",
)
sft_model = PeftModel.from_pretrained(base_model, SFT_CHECKPOINT)
merged    = sft_model.merge_and_unload()   # bake SFT weights in permanently
print("SFT weights merged.")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
policy = get_peft_model(merged, lora_config)
policy.config.use_cache = False
policy.print_trainable_parameters()

In [ ]:
# ── DPO training
use_bf16 = (torch_dtype == torch.bfloat16)
use_fp16 = (torch_dtype == torch.float16)

dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    logging_dir=DPO_LOGGING_DIR,
    beta=BETA,
    loss_type="sigmoid",
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    optim="adamw_torch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="no",
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

trainer = DPOTrainer(
    model=policy,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    max_prompt_length=MAX_PROMPT_LEN,
    max_length=MAX_LEN,
)

print("Starting DPO training...")
train_result = trainer.train()

trainer.save_model(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)
trainer.save_state()

train_metrics = train_result.metrics
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
print("Training complete. Metrics:", train_metrics)

In [ ]:
# Save final summary
summary = {
    "variant": "dpo",
    "student_model": STUDENT_MODEL_ID,
    "sft_checkpoint": SFT_CHECKPOINT,
    "num_dpo_pairs": len(dpo_pairs),
    "beta": BETA,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "train_metrics": train_metrics,
}
with open(f"{DPO_OUTPUT_DIR}/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Adapter saved → {DPO_OUTPUT_DIR}")
print(json.dumps(summary, indent=2))